In [1]:
import json
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

DATA_DIR = "sepsis_mca_dataset_v2"

train_traces = json.load(open(f"{DATA_DIR}/X_train.json"))
test_traces  = json.load(open(f"{DATA_DIR}/X_test.json"))

df = pd.read_csv(f"{DATA_DIR}/y_test.csv")
test_labels = df["label"].values

print("Train:", len(train_traces))
print("Test:", len(test_traces))

Train: 840
Test: 198


In [2]:
train_variants = [tuple(t) for t in train_traces]
test_variants  = [tuple(t) for t in test_traces]

In [3]:
variant_counts = Counter(train_variants)

print("Unique variants in train:", len(variant_counts))

Unique variants in train: 680


In [4]:
def variant_score(variant):
    freq = variant_counts.get(variant, 0)
    
    # negative log frequency (more standard)
    return -np.log((freq + 1) / (len(train_variants) + 1))

In [5]:
train_scores = np.array([
    variant_score(v) for v in train_variants
])

test_scores = np.array([
    variant_score(v) for v in test_variants
])

In [6]:
threshold = np.percentile(train_scores, 70)

preds = (test_scores >= threshold).astype(int)

In [7]:
print("Confusion Matrix:")
print(confusion_matrix(test_labels, preds))

print("\nClassification Report:")
print(classification_report(test_labels, preds))

auc = roc_auc_score(test_labels, test_scores)
print("ROC-AUC:", auc)

Confusion Matrix:
[[ 15 126]
 [  1  56]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.11      0.19       141
           1       0.31      0.98      0.47        57

    accuracy                           0.36       198
   macro avg       0.62      0.54      0.33       198
weighted avg       0.76      0.36      0.27       198

ROC-AUC: 0.5765210899589398
